# 09 - A candidate run: training, fit and backtest

One saved run, end to end: what it is, its training record, how its heads score on the out-of-sample
(dev) block against the baselines, and a backtest at the notebook's own strategy and costs. Everything
here reads the run's saved files (config, metrics, the evaluation report, the saved prediction blocks)
so it needs neither the training CSV nor a fresh inference pass (`neural_trade.notebook.run_report`).

The default `RUN_DIR` is candidate **C1** (owner "сохрани 1-3", `configs/candidates/README.md`): the
360-day model's fold -2, seed 0 cell, `calibrated_quantile` at `entry_quantile` 0.9, full size, at
**zero trading costs** (D-044). Its dev (out-of-sample) block is 2025-07-27 to 2025-08-28.

In [1]:
# Parameters
ROOT = ".."
RUN_DIR = "../runs/scenarios/long_360d_stab/20260930T094257Z-dce15ed-e3669618-default__f-2__s0"
CANDIDATE_ID = "C1"
MANIFEST_PATH = "../configs/candidates/manifest.json"
STRATEGY = "calibrated_quantile"
STRATEGY_PARAMS = {"entry_quantile": 0.9}
BACKTEST_PARAMS = {"fee_bps": 0.0, "half_spread_bps": 0.0, "slippage_bps": 0.0, "random_seeds": 20}
MAX_DASHBOARD_POINTS = 6000      # thins the trading dashboard's per-bar lines on this long a block (D-013 size)
MAX_ROC_POINTS = 300             # direction_analytics' own ROC-curve resolution

## Summary

In [2]:
from pathlib import Path

from IPython.display import HTML, Markdown, display

from neural_trade.notebook import BacktestExplorer, run_report
from neural_trade.visualization import analytics_tables as AT

run_dir = Path(RUN_DIR)
print("run:", run_dir)
display(Markdown(run_report.overview_markdown(run_dir, candidate_id=CANDIDATE_ID, manifest_path=MANIFEST_PATH)))
run_report.blocks_table(run_dir)

run: ..\runs\scenarios\long_360d_stab\20260930T094257Z-dce15ed-e3669618-default__f-2__s0


**20260930T094257Z-dce15ed-e3669618-default__f-2__s0**  (commit `dce15ed`, scenario `long_360d_stab`, cell `default__f-2__s0`, fold -2, seed 0)

- dataset `Bitcoin_BTCUSDT.csv`, 1-minute bars, 60-bar window, horizons 10m/15m/20m
- served weights: epoch 4 (val loss 5.9269, best validation epoch, restored by EarlyStopping (stopped at epoch 10))
- 10 epochs trained, 0.3087 sec/step
- default strategy `calibrated_quantile`
- candidate `C1` (rank 1, per_model, params {'entry_quantile': 0.9, 'size': 1.0}): mean return +10.17% over 6/6 profitable cells, mean win share 54.1%, max drawdown 6.45%

,n bars,first,last
block,,,
train,518432,2024-06-01T08:18:00,2025-05-28T04:09:00
val,43200,2025-05-28T05:30:00,2025-06-27T05:29:00
cal,43200,2025-06-27T06:50:00,2025-07-27T06:49:00
test,46544,2025-07-27T08:10:00,2025-08-28T15:53:00


### Key numbers (dev block): direction AUC against the logistic-regression-on-lags baseline, variance CRPSS against constant variance, and 0.90-target conformal coverage.

In [3]:
display(AT.styled(run_report.key_numbers_table(run_dir)))

,direction AUC (model),direction AUC (logreg_lags),variance CRPSS vs constant,coverage @ 0.90
h0,0.5196,0.5243,0.0457,0.9072
h1,0.5184,0.5272,0.0437,0.9081
h2,0.5245,0.5314,0.0435,0.9079


### The candidate's six cells, for context (`configs/candidates/manifest.json`; already verified, not recomputed here)

In [4]:
run_report.candidate_cells_table(CANDIDATE_ID, MANIFEST_PATH).round(4)

,return,win_share,max_drawdown,sharpe_net,trades,buy_and_hold
cell,,,,,,
20260930T081536Z-dce15ed-e1204c70-default__f-3__s0,0.0286,0.5140,0.0558,1.63,1825,0.1127
20260930T084838Z-dce15ed-10eeccf2-default__f-3__s1,0.1474,0.5254,0.0269,7.49,1656,0.1127
20260930T091333Z-dce15ed-23f3fd33-default__f-3__s2,0.0313,0.5393,0.0371,1.72,1398,0.1127
20260930T094257Z-dce15ed-e3669618-default__f-2__s0,0.1671,0.5467,0.0466,7.82,1968,-0.0485
20260930T095943Z-dce15ed-72410bdc-default__f-2__s1,0.0626,0.5670,0.0645,3.10,1559,-0.0485
20260930T102411Z-dce15ed-b5401d57-default__f-2__s2,0.1731,0.5533,0.0521,7.42,2026,-0.0485


## Training

The training dashboard (every logged validation metric per horizon with its chance range) and the loss
terms, from the run's `metrics.jsonl`; then the learned indicator periods over training (the per-window
applied range needs the training CSV and is not shown here; see notebook 07 on a run with it at hand).

In [5]:
from neural_trade.registries.visualizations import Visualizations
from neural_trade.visualization.indicator_evolution import indicator_summary
from neural_trade.visualization.training_dashboard import training_health_html

cfg = run_report.load_config(run_dir)
metrics_path = run_dir / "metrics.jsonl"
display(HTML(training_health_html(metrics_path, cfg)))
Visualizations.build("training_dashboard", metrics_path, cfg).show()
Visualizations.build("training_loss_terms", metrics_path, cfg).show()

term,weight in the total,epoch 4 (served): val,share of val,train,epoch 10 (last): val
point,λ 0.644 / 1.7 / 1.54 (inside),1.021,17.2%,1.211,1.019
trend,λ_ext 0.1 (inside) × 1,0.07409,1.2%,0.08951,0.07214
direction BCE,λ_dir 0.749 × 1,1.558,26.3%,1.564,1.558
NLL,λ_var 0.382 × 1,1.271,21.5%,1.311,1.284
CRPS,λ_crps 0.896,1.144,19.3%,1.241,1.152
soft ECE,λ_soft_ece 2.91,0.4669,7.9%,0.2256,0.4524
volatility,λ_vol 1.82 (inside) × 0.1,0.1358,2.3%,0.1403,0.141
physics,λ 0.1 (inside),0.03072,0.5%,0.1103,0.05946
T-perp,,0.01598,,0.006931,0.04174
Casimir,,8.003e-05,,0.0004267,3.802e-05


In [6]:
Visualizations.build("indicator_evolution", metrics_path, cfg).show()
display(AT.styled(indicator_summary(metrics_path, cfg)))

,indicator,start,after epoch 1,last,min,max,change %,slope %/epoch (last 5),headroom %,near bound,CV,corr with val loss,r with epoch,r of changes
macd_1_fast,MACD #1 fast,5.0000,3.8469,2.6596,2.0971,5.0000,-46.8075,1.5791,32.9812,,0.1903,0.6988,-0.7995,0.3569
ma_period_0,MA #0,5.0000,4.1805,3.4371,2.9466,5.0000,-31.2579,1.0390,71.8552,,0.1067,0.6717,-0.8015,0.3085
bb_period_0,BB #0,10.0000,9.0227,7.0341,6.1184,10.0000,-29.6586,1.2124,251.7072,,0.1227,0.6765,-0.7970,0.2585
ma_period_1,MA #1,10.0000,8.6687,7.5365,6.6047,10.0000,-24.6349,0.9770,276.8252,,0.0857,0.6329,-0.8050,0.2063
macd_2_fast,MACD #2 fast,8.0000,7.4830,6.0343,4.9639,8.0000,-24.5708,1.2576,201.7169,,0.1277,0.6648,-0.8085,0.2841
macd_0_fast,MACD #0 fast,12.0000,11.3521,9.2086,8.2993,12.0000,-23.2614,1.0352,360.4316,,0.0994,0.6923,-0.8016,0.3759
macd_1_slow,MACD #1 slow,35.0000,41.3466,42.9472,35.0000,47.1854,22.7063,-1.3839,39.7065,,0.0485,-0.4641,0.6992,-0.0385
bb_period_2,BB #2,25.0000,24.5054,19.5915,17.9439,25.0000,-21.6339,0.6262,206.2548,,0.0962,0.6956,-0.7889,0.2902
macd_2_signal,MACD #2 signal,9.0000,8.3166,7.2818,6.2657,9.0000,-19.0908,0.9517,264.0913,,0.0859,0.6282,-0.7900,0.1786
rsi_period_1,RSI #1,14.0000,12.4887,11.7026,11.4404,14.0000,-16.4100,0.1623,412.7066,,0.0253,0.6657,-0.7414,0.1471


## Fit evaluation on the out-of-sample block

The evaluation report already scored on the run (`eval_report_dev.md`/`.json`) against every baseline;
then the per-horizon direction, price and variance diagnostics notebook 04 draws, and the reliability /
coverage tables, all from the saved `predictions_oos.npz`.

In [7]:
from neural_trade.evaluation.report import EvalReport
from neural_trade.experiments.scorer import load_block

report = EvalReport.from_json(run_dir / "eval_report_dev.json")
display(Markdown(report.to_markdown()))
display(AT.styled(AT.baseline_table(report)))
oos_frame, oos_bars, oos_extra = load_block(run_dir / "predictions_oos.npz")
raw_delta = oos_frame.meta.get("delta_raw")
display(AT.styled(AT.classification_table(oos_frame, cfg)))
display(AT.styled(AT.delta_quality_table(oos_frame, cfg, raw_delta=raw_delta)))

# Evaluation report - dev split - run `20260930T094257Z-dce15ed-e3669618-default__f-2__s0`

n = 46544 samples, one per 1-minute bar. Direction metrics count only moves beyond 5 bps (the neutral mask). Consecutive samples share most of their target window, so n_eff = n // bars ahead counts the non-overlapping outcomes.

## Direction heads: P(up) > 0.5 on moves beyond 5 bps

Up is the positive class. Temperature scaling does not move P(up) across 0.5, so the counts and rates are the same before and after calibration; Brier and ECE are not.

| metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|
| n scored (outside the deadband) | 28800 | 32081 | 33877 |
| n_eff of the scored moves (n scored // bars ahead) | 2880 | 2138 | 1693 |
| true up-rate | 0.4906 | 0.4906 | 0.4890 |
| calls up (predicted up-rate) | 0.4653 | 0.7152 | 0.6899 |
| accuracy | 0.5193 | 0.5043 | 0.5095 |
| balanced accuracy | 0.5187 | 0.5084 | 0.5137 |
| precision (up) | 0.5106 | 0.4965 | 0.4989 |
| recall / sensitivity (up) | 0.4844 | 0.7237 | 0.7040 |
| specificity (down) | 0.5530 | 0.2930 | 0.3235 |
| F1 (up) | 0.4971 | 0.5889 | 0.5840 |
| MCC | 0.0374 | 0.0186 | 0.0297 |
| AUC | 0.5196 | 0.5184 | 0.5245 |
| Brier | 0.2502 | 0.2499 | 0.2503 |
| ECE (positive class) | 0.0111 | 0.0177 | 0.0266 |
| ECE of a constant 0.5 (= distance of the up-rate from 0.5) | 0.0094 | 0.0094 | 0.0110 |
| TP / FP / TN / FN | 6843 / 6559 / 8113 / 7285 | 11391 / 11553 / 4789 / 4348 | 11661 / 11711 / 5601 / 4904 |
| Gaussian readout: calls up | 0.8649 | 0.7865 | 0.7533 |
| Gaussian readout: MCC | 0.0072 | 0.0193 | 0.0011 |
| Gaussian readout: AUC | 0.5031 | 0.5111 | 0.5071 |
| Gaussian readout: Brier | 0.2502 | 0.2499 | 0.2500 |
| Gaussian readout: ECE | 0.0162 | 0.0127 | 0.0121 |

## Price heads (dollars)

| metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|
| RMSE ($), served | 164.27 | 198.27 | 228.32 |
| RMSE ($), raw heads | 164.44 | 203.18 | 228.84 |
| RMSE ($), zero prediction | 164.21 | 198.38 | 228.36 |
| MAE ($), served | 112.69 | 137.97 | 158.98 |
| MAE ($), raw heads | 112.83 | 139.26 | 159.26 |
| MAE ($), zero prediction | 112.64 | 138.04 | 158.99 |
| skill vs zero (1 - MSE / MSE of 0), served | -0.0007 | 0.0011 | 0.0003 |
| skill vs zero, raw heads | -0.0027 | -0.0489 | -0.0042 |
| EV, served | -0.0004 | 0.0013 | 0.0004 |
| EV, raw heads | -0.0017 | -0.0463 | -0.0022 |
| corr, Pearson, raw heads (the same for served while beta > 0) | -0.0039 | 0.0363 | 0.0345 |
| corr, Spearman, raw heads | 0.0077 | 0.0183 | 0.0066 |
| mean predicted ($), served | 1.88 | 1.43 | 0.55 |
| mean predicted ($), raw heads | 4.32 | 8.56 | 8.07 |
| mean realised ($) | -1.18 | -1.76 | -2.32 |
| share predicted up, raw heads | 0.8642 | 0.8001 | 0.7438 |
| share realised up | 0.4897 | 0.4876 | 0.4888 |
| shrink beta (served = beta x raw, fit on cal) | 0.4356 | 0.1666 | 0.0684 |

Correlations and the share predicted up are the raw heads': the served delta, beta x raw, has the same while beta > 0.

## Variance heads

| metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|
| CRPS ($) | 82.08 | 100.22 | 115.52 |
| CRPSS vs constant variance | 0.0457 | 0.0437 | 0.0435 |
| NLL | 6.4015 | 6.5935 | 6.7376 |
| PIT KS | 0.0446 | 0.0433 | 0.0441 |
| var / err^2 Spearman | 0.3359 | 0.3300 | 0.3348 |
| coverage of the 90% interval | 0.9072 | 0.9081 | 0.9079 |
| width of the 90% interval ($) | 519.12 | 635.23 | 729.30 |

## Confidence gap (accuracy of the more confident half minus the less confident half)

| horizon | gap | 95% CI | verdict |
|---|---|---|---|
| h0 | 0.0118 | [-0.0049, 0.0287] | NOISE |
| h1 | 0.0130 | [-0.0061, 0.0315] | NOISE |
| h2 | 0.0203 | [-0.0015, 0.0429] | NOISE |

## Coherence across horizons

Magnitude ordering: the share of samples whose predicted move grows with the horizon, as the loss asks. Magnitudes in random order would give 0.5, 0.5 and 0.1667.

| check | raw price heads (the trained ordering) | served deltas (beta-shrunk: h0 0.436 / h1 0.167 / h2 0.068) | realised moves |
|---|---|---|---|
| abs(d h0) <= abs(d h1) | 0.5989 | 0.3159 | 0.6180 |
| abs(d h1) <= abs(d h2) | 0.5726 | 0.3179 | 0.5923 |
| full chain h0 <= h1 <= h2 | 0.2512 | 0.0347 | 0.3360 |

The served ordering mostly reflects the per-horizon shrink beta, not the model: judge the trained constraint on the raw heads.

Sign agreement: sign(raw price head) against calibrated P(up) > 0.5 (the served delta, beta x raw, has the same sign while beta > 0).

| | h0 | h1 | h2 | all 3 |
|---|---|---|---|---|
| agree | 0.4830 | 0.7418 | 0.5326 | 0.2212 |
| expected if the two signs were independent | 0.4671 | 0.6382 | 0.5857 | 0.1900 |

- P(up) unanimity (all three horizons call the same side): 0.3583

## Against baselines (fit on the training block)

Each cell: model vs baseline, the margin (positive = the model is better) and the verdict. "noise": |z| < 1.96, so the ordering is not established; "significantly worse": the model loses with z <= -1.96. "DM z": Diebold-Mariano test of the per-sample loss difference (RMSE and skill, MAE, Brier, accuracy, CRPS, NLL) with a Bartlett (Newey-West) long-run variance, lag 2 x bars ahead. "boot z": the margin over its standard error in a paired moving-block bootstrap (80-bar blocks, 500 resamples; MCC, AUC, balanced accuracy, ECE, EV, corr, PIT KS, var / err^2 Spearman). Rows that only restate the RMSE verdict for a constant prediction (EV, corr and skill against zero_delta / mean_delta) are left out; the JSON keeps every verdict.

| baseline | metric | h0 (10 bars) | h1 (15 bars) | h2 (20 bars) |
|---|---|---|---|---|
| logreg_lags | direction/mcc | 0.0374 vs 0.0344 (+0.0030): beats, noise (boot z +0.19) | 0.0186 vs 0.0378 (-0.0192): does not beat, noise (boot z -1.33) | 0.0297 vs 0.0411 (-0.0114): does not beat, noise (boot z -0.99) |
| logreg_lags | direction/auc | 0.5196 vs 0.5243 (-0.0047): does not beat, noise (boot z -0.46) | 0.5184 vs 0.5272 (-0.0088): does not beat, noise (boot z -1.02) | 0.5245 vs 0.5314 (-0.0069): does not beat, noise (boot z -0.95) |
| logreg_lags | direction/brier | 0.2502 vs 0.2497 (-0.0005): does not beat, noise (DM z -1.12) | 0.2499 vs 0.2497 (-0.0001): does not beat, noise (DM z -0.46) | 0.2503 vs 0.2497 (-0.0006): does not beat, noise (DM z -1.36) |
| logreg_lags | direction/ece_pos | 0.0111 vs 0.0112 (+0.0001): beats, noise (boot z +0.03) | 0.0177 vs 0.0116 (-0.0061): does not beat, noise (boot z -1.01) | 0.0266 vs 0.0137 (-0.0128): does not beat, noise (boot z -1.77) |
| logreg_lags | direction/acc | 0.5193 vs 0.5165 (+0.0028): beats, noise (DM z +0.36) | 0.5043 vs 0.5178 (-0.0134): does not beat, noise (DM z -1.85) | 0.5095 vs 0.5191 (-0.0096): does not beat, noise (DM z -1.50) |
| logreg_lags | direction/bal_acc | 0.5187 vs 0.5171 (+0.0015): beats, noise (boot z +0.19) | 0.5084 vs 0.5188 (-0.0104): does not beat, noise (boot z -1.49) | 0.5137 vs 0.5204 (-0.0067): does not beat, noise (boot z -1.17) |
| class_prior | direction/mcc | 0.0374 vs 0.0000 (+0.0374): beats (boot z +3.33) | 0.0186 vs 0.0000 (+0.0186): beats, noise (boot z +1.53) | 0.0297 vs 0.0000 (+0.0297): beats (boot z +2.20) |
| class_prior | direction/auc | 0.5196 vs 0.5000 (+0.0196): beats (boot z +2.69) | 0.5184 vs 0.5000 (+0.0184): beats (boot z +2.15) | 0.5245 vs 0.5000 (+0.0245): beats (boot z +2.61) |
| class_prior | direction/brier | 0.2502 vs 0.2501 (-0.0002): does not beat, noise (DM z -0.39) | 0.2499 vs 0.2501 (+0.0002): beats, noise (DM z +1.03) | 0.2503 vs 0.2501 (-0.0002): does not beat, noise (DM z -0.37) |
| class_prior | direction/ece_pos | 0.0111 vs 0.0123 (+0.0012): beats, noise (boot z +0.32) | 0.0177 vs 0.0134 (-0.0043): does not beat, significantly worse (boot z -2.58) | 0.0266 vs 0.0164 (-0.0102): does not beat, significantly worse (boot z -3.66) |
| class_prior | direction/acc | 0.5193 vs 0.4906 (+0.0287): beats (DM z +2.91) | 0.5043 vs 0.4906 (+0.0137): beats, noise (DM z +1.84) | 0.5095 vs 0.4890 (+0.0206): beats (DM z +2.33) |
| class_prior | direction/bal_acc | 0.5187 vs 0.5000 (+0.0187): beats (boot z +3.33) | 0.5084 vs 0.5000 (+0.0084): beats, noise (boot z +1.53) | 0.5137 vs 0.5000 (+0.0137): beats (boot z +2.20) |
| zero_delta | delta/rmse | 164.27 vs 164.21 (-0.06, -0.03%): does not beat, noise (DM z -0.98) | 198.27 vs 198.38 (+0.11, +0.06%): beats, noise (DM z +0.51) | 228.32 vs 228.36 (+0.04, +0.02%): beats, noise (DM z +0.70) |
| zero_delta | delta/mae | 112.69 vs 112.64 (-0.05, -0.05%): does not beat, noise (DM z -1.55) | 137.97 vs 138.04 (+0.07, +0.05%): beats, noise (DM z +0.76) | 158.98 vs 158.99 (+0.02, +0.01%): beats, noise (DM z +0.66) |
| mean_delta | delta/rmse | 164.27 vs 164.22 (-0.05, -0.03%): does not beat, noise (DM z -0.92) | 198.27 vs 198.40 (+0.12, +0.06%): beats, noise (DM z +0.58) | 228.32 vs 228.38 (+0.06, +0.03%): beats, noise (DM z +1.18) |
| mean_delta | delta/mae | 112.69 vs 112.65 (-0.03, -0.03%): does not beat, noise (DM z -1.26) | 137.97 vs 138.07 (+0.10, +0.07%): beats, noise (DM z +1.12) | 158.98 vs 159.03 (+0.06, +0.03%): beats, noise (DM z +1.94) |
| const_var | variance/crps | 82.08 vs 86.01 (+3.93, +4.57%): beats (DM z +21.02) | 100.22 vs 104.80 (+4.58, +4.37%): beats (DM z +16.63) | 115.52 vs 120.77 (+5.25, +4.35%): beats (DM z +15.06) |
| const_var | variance/nll | 6.4015 vs 6.5314 (+0.1298): beats (DM z +8.07) | 6.5935 vs 6.7225 (+0.1290): beats (DM z +8.86) | 6.7376 vs 6.8636 (+0.1260): beats (DM z +8.45) |
| const_var | variance/pit_ks | 0.0446 vs 0.0915 (+0.0469): beats (boot z +12.37) | 0.0433 vs 0.0894 (+0.0460): beats (boot z +12.70) | 0.0441 vs 0.0910 (+0.0470): beats (boot z +12.42) |
| const_var | variance/corr_var_err2_spearman | 0.3359 vs 0.0000 (+0.3359): beats (boot z +24.51) | 0.3300 vs 0.0000 (+0.3300): beats (boot z +22.23) | 0.3348 vs 0.0000 (+0.3348): beats (boot z +21.75) |

## Backtest (costs included)

- n_trades: 1968
- total_return: -0.9930
- sharpe_net: -160.1782
- sharpe_gross: 7.5855
- sortino: -172.1878
- max_drawdown: 0.9930
- hit_rate: 0.0305
- hit_rate_gross: 0.5467
- profit_factor: 0.0109
- avg_hold_bars: 8.3186
- exposure: 0.3518
- turnover: 795.5989
- fees_paid: 7956.2802
- traded_notional: 7956280.2320
- breakeven_cost_bps: 1.0374
- gross_edge_per_trade_bps: 0.7962
- costs_paid: 10343.1643
- gross_pnl: 412.6918
- net_pnl: -9930.4725


horizon,h0,h1,h2
metric,,,
bars ahead,10,15,20
n samples,46544,46544,46544
n scored (moves beyond 5 bps),28800,32081,33877
n_eff of the scored moves (n scored // bars ahead),2880,2138,1693
true up-rate,0.4906,0.4906,0.4890
calls up (predicted up-rate),0.4653,0.7152,0.6899
accuracy,0.5193,0.5043,0.5095
accuracy 95% CI (n_eff),"[0.5010, 0.5375]","[0.4832, 0.5255]","[0.4857, 0.5333]"
majority-class accuracy (hindsight),0.5094,0.5094,0.5110


horizon,h0,h1,h2
metric,,,
bars ahead,10,15,20
n samples,46544,46544,46544
n_eff (non-overlapping outcomes),4654,3102,2327
shrink beta (served = beta x raw),0.4356,0.1666,0.0684
"RMSE ($), raw heads",164.44,203.18,228.84
"RMSE ($), served",164.27,198.27,228.32
"RMSE ($), zero prediction",164.21,198.38,228.36
"MAE ($), raw heads",112.83,139.26,159.26
"MAE ($), served",112.69,137.97,158.98


In [8]:
Visualizations.build("direction_analytics", oos_frame, cfg, raw_delta=raw_delta, max_points=MAX_ROC_POINTS).show()
Visualizations.build("delta_analytics", oos_frame, cfg, raw_delta=raw_delta).show()
Visualizations.build("variance_analytics", oos_frame, cfg, raw_delta=raw_delta).show()

## Backtest

`calibrated_quantile` at `entry_quantile` 0.9, fitted on the calibration block, at zero trading costs
(D-044): the summary against buy-and-hold, always-flat and the size-matched random null, the trading
dashboard (equity vs buy-and-hold, drawdown, positions; its per-bar lines thinned to `MAX_DASHBOARD_POINTS`
on a block this long, every trade and decision marker still at its own bar) and the per-trade analytics.

In [9]:
explorer = BacktestExplorer(run_report.load_saved_blocks(run_dir))
res = explorer.run(STRATEGY, STRATEGY_PARAMS, BACKTEST_PARAMS)
display(explorer.summary_frame(res, styled=True))
explorer.dashboard(res, max_line_points=MAX_DASHBOARD_POINTS).show()
explorer.trade_analytics(res).show()

CalibrationPipeline loaded from '..\runs\scenarios\long_360d_stab\20260930T094257Z-dce15ed-e3669618-default__f-2__s0\artifacts\calibration/'


,calibrated_quantile,buy_and_hold,always_flat,"random same freq (mean of 20 seeds, size 1.00)"
trades,"1,968",1,0,–
net return (after costs),+16.71%,-4.85%,0.00%,+0.27%
gross return (before costs),+16.71%,-4.85%,0.00%,+0.27%
costs paid ($),0,0,0,–
of which fees ($),0,0,0,–
Sharpe after costs (annualised),+7.82,-1.54,0.00,+0.18
max drawdown,4.66%,12.55%,0.00%,–
hit rate after costs,54.7%,0.0%,–,–
hit rate before costs,54.7%,0.0%,–,–
profit factor,1.16,0.00,–,–
